# 🇲🇲 AI Voice Studio — Phone Only Myanmar Voice Clone
Use this notebook from your phone in Google Colab. Choose a GPU runtime, run all cells, then open the Gradio public link. The phone is only the control device; VoxCPM2 runs on the temporary GPU runtime.

Features: Myanmar-only voice cloning • reference audio • up to 5,000 characters • speed 0–100 • WAV download.

Speed control accepts every integer value from 0 through 100.

Note: Colab free GPU availability and runtime duration are not guaranteed or unlimited.

In [ ]:
!pip -q install -U voxcpm gradio librosa soundfile
import torch
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU required. In Colab: Runtime → Change runtime type → GPU.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))

In [ ]:
from voxcpm import VoxCPM
import gradio as gr
import numpy as np
import librosa
import soundfile as sf
import os, tempfile

MODEL_ID = 'openbmb/VoxCPM2'
model = VoxCPM.from_pretrained(MODEL_ID, load_denoiser=False)
SR = int(getattr(model.tts_model, 'sample_rate', 48000))
print('VoxCPM2 ready. Sample rate:', SR)

In [ ]:
def has_myanmar(text):
    return any('\u1000' <= c <= '\u109f' or '\uaa60' <= c <= '\uaa7f' for c in text)

def clone_voice(reference_audio, text, speed_percent):
    if not reference_audio:
        raise gr.Error('Reference voice audio ထည့်ပါ။')
    text = (text or '').strip()
    if not text:
        raise gr.Error('မြန်မာစာသား ထည့်ပါ။')
    if len(text) > 5000:
        raise gr.Error('စာသား 5,000 characters ထက် မကျော်ရပါ။')
    if not has_myanmar(text):
        raise gr.Error('Myanmar/Burmese text ပဲ ထည့်ပါ။')
    speed_percent = int(max(0, min(100, int(speed_percent))))
    speed = 0.5 + (speed_percent / 100.0) * 1.5
    wav = model.generate(text=text, reference_wav_path=reference_audio, cfg_value=2.0, inference_timesteps=10)
    audio = wav.detach().float().cpu().numpy() if hasattr(wav, 'detach') else np.asarray(wav)
    audio = np.squeeze(audio)
    if abs(speed - 1.0) > 0.001:
        audio = librosa.effects.time_stretch(audio, rate=speed)
    out = os.path.join(tempfile.gettempdir(), 'myanmar-voice-clone.wav')
    sf.write(out, audio, SR)
    return out

In [ ]:
with gr.Blocks(title='AI Voice Studio • Myanmar Voice Clone', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🇲🇲 AI Voice Studio\n### Myanmar Voice Clone • Phone Only')
    gr.Markdown('Reference voice တင်ပြီး မြန်မာစာသားကို အဲဒီအသံပုံစံနဲ့ ထုတ်ပါ။ Maximum 5,000 characters • Speed 0–100 (integer)')
    ref = gr.Audio(label='🎙️ Reference Voice', type='filepath')
    text = gr.Textbox(label='မြန်မာစာသား', placeholder='မြန်မာစာသားရေးပါ...', lines=8, max_lines=12, max_length=5000)
    speed = gr.Slider(0, 100, value=50, step=1, label='Speed (0–100)', info='Every integer from 0 to 100 is available.')
    go = gr.Button('▶ Myanmar Voice Clone', variant='primary')
    output = gr.Audio(label='🔊 Myanmar Output', type='filepath')
    go.click(clone_voice, inputs=[ref, text, speed], outputs=output)

demo.launch(share=True, debug=False)